<!-- nav -->
[← 13 · Analyzing and rewriting constraints](13_Analyzing_And_Rewriting_Constraints.ipynb) · [Home](../../README.md) · [15 · Constraints for firmware and hardware →](15_Constraints_For_Firmware_And_Hardware.ipynb)

# 14 · One constraint, many languages

**The problem.** Back to the four copies of case study 9. The QA team checks shipments in Excel; the data pipeline is
Python; the analysts use MATLAB; the specification is a LaTeX document. Each wants the constraint in its own language,
and each copy, written by hand, drifts.

mbse-expressions models each of those languages as a **dialect**: its expressions are data too, it has its own folding rules, and it writes its own source text. **Translators** between dialects turn the neutral constraint (in the Patterns
dialect, which the earlier case studies used) into each, and back.

In [1]:
import { Simplifiers, Expressions as E } from "@mbse/expressions/Patterns/Expressions";
import * as Excel from "@mbse/expressions/Excel/Expressions";
import * as Latex from "@mbse/expressions/LaTeX/Expressions";
import * as Matlab from "@mbse/expressions/MATLAB/Expressions";
import * as Python3 from "@mbse/expressions/Python/Python3";
import { Terms } from "@mbse/expressions/Framework";
import * as Translators from "@mbse/expressions/Translators";
import { cold_chain, show } from "./toolkit.js";

const store = cold_chain(); // with each dialect's kinds installed, so that it holds their expressions too
const [Expression, Term] = [store.Expressions.Framework.Expression, store.Expressions.Framework.Term];
const [shipment, c] = [E.variable("shipment"), E.variable("c")];
const [in_range, allowed] = store.transaction(() => [
  Expression((e: any) => e.root(E.literal(2.0).le(shipment.celsius).and_(shipment.celsius.le(8.0)))),
  Expression((e: any) => e.root(shipment.readings.count_where("c", c.gt(8.0)).le(2n).and_(shipment.readings.max().le(12.0))))]);
const languages: [any, any][] = [[Excel.Expressions.DIALECT, Excel.Text], [Python3.Expressions.DIALECT, Python3.Text],
  [Matlab.Expressions.DIALECT, Matlab.Text], [Latex.Expressions.DIALECT, Latex.Text]];

/** `constraint`, of the dialect `source`, translated to `target`: the translation is a writer, written as a term. */
function translate(constraint: unknown, source: any, target: any): any {
  return store.transaction(() => Term(Translators.between(source, target).forward(constraint)));
}

<a id="T14.1"></a>
## 14.1 · Translating

`Translators.between(source, target)` finds the translator, and `forward` translates, giving a writer of the target
dialect's term, which `translate` (above) writes in a transaction, as any term is written. Each dialect's `Text.ToText`
writes its source:

In [2]:
for (const [dialect, text] of languages) {
  console.log(dialect.name().padEnd(7), text.ToText(translate(in_range, E.DIALECT, dialect)));
}

Excel   =AND(2.0 <= shipment.celsius, shipment.celsius <= 8.0)


Python3 2.0 <= shipment.celsius and shipment.celsius <= 8.0


MATLAB  2.0 <= shipment.celsius && shipment.celsius <= 8.0


LaTeX   2.0 \leq \mathit{shipment}.\mathit{celsius} \land \mathit{shipment}.\mathit{celsius} \leq 8.0


Collections translate to each language's idiom: `MAP` and `LAMBDA` in Excel, generator expressions in Python,
`arrayfun` in MATLAB. LaTeX has no counterpart for a quantifier yet, a known gap
([CAP-102: collections translated to LaTeX and C and C++](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-102.md)), and says so:

In [3]:
for (const [dialect, text] of languages) {
  try {
    console.log(dialect.name().padEnd(7), text.ToText(translate(allowed, E.DIALECT, dialect)));
  } catch (error) {
    console.log(dialect.name().padEnd(7), (error as Error).message);
  }
}

Excel   =AND(SUM(MAP(shipment.readings, LAMBDA(c, IF(c > 8.0, 1, 0)))) <= 2, MAX(shipment.readings) <= 12.0)


Python3 sum(1 for c in shipment.readings if c > 8.0) <= 2 and max(shipment.readings) <= 12.0


MATLAB  nnz(arrayfun(@(c) c > 8.0, shipment.readings)) <= 2 && max(shipment.readings) <= 12.0


LaTeX   Patterns quantifier 'count' has no LaTeX counterpart


<a id="T14.2"></a>
## 14.2 · Each language simplifies by its own rules

A translation keeps the constraint's **form**. Its value is the target language's business, because that's where it will
run. Simplify the translations with a shipment whose logger never reported:

In [4]:
const silent = store.transaction(() => store.Shipment((s: any) => s.id("S3")));
const excel = translate(in_range, E.DIALECT, Excel.Expressions.DIALECT);
const python = translate(in_range, E.DIALECT, Python3.Expressions.DIALECT);
const flagged = store.transaction(() => Term(Excel.Expressions.function_("ISERROR", excel))); // an error has no constant: ask whether it is one
console.log("Patterns", Simplifiers.OfAny(in_range, { shipment: silent }).value);
console.log("Excel   ", "ISERROR:", String(Excel.Simplifiers.OfAny(flagged, { shipment: silent }).evaluate()));
try {
  Python3.Simplifiers.OfAny(python, { shipment: silent }).evaluate();
} catch (error) {
  console.log("Python3 ", String(error));
}

Patterns null


Excel    #FIELD!


Python3  AttributeError: 'Shipment' object has no attribute 'celsius'


Patterns says unknown; Excel gives the error value `#FIELD!`, which no constant spells, so the formula stays, and
propagates through a worksheet the way errors do there (`ISERROR` of it folds to TRUE); Python raises, as
`shipment.celsius` would. Coercion differs too. Excel compares an integer limit with a float
reading happily, where Patterns says unknown (case study 10):

In [5]:
const [sloppy, cold] = store.transaction(() => [
  Expression((e: any) => e.root(shipment.celsius.ge(2n))), store.Shipment((s: any) => s.id("S1").celsius(5.5))]);
console.log("Patterns", Simplifiers.OfAny(sloppy, { shipment: cold }).value);
console.log("Excel   ", Excel.Simplifiers.OfAny(translate(sloppy, E.DIALECT, Excel.Expressions.DIALECT), { shipment: cold }).evaluate());

Patterns null


Excel    true


So keep the constraint in Patterns, where its meaning is fixed, translate it to where it is checked, and test it there with that language's simplifier, whose rules model its own exactly.

<a id="T14.3"></a>
## 14.3 · Reading constraints back

A constraint written in another language can be read into Patterns, to analyze it (case study 13) or store it. Python's
dialect builds its expressions as Python's `ast` module has them (`compare`, `attribute`, `boolop` and so on), and the
translator goes the other way. (In Python, `Text.FromText` also parses source, as [tutorial 3](03_The_Smallest_Expression_A_Literal.ipynb#T03.2) shows.)

In [6]:
const P = Python3.Expressions;
const celsius = P.attribute(P.name("shipment"), "celsius");
const parsed = store.transaction(() => Term(P.boolop("and", P.compare("<=", 2.0, celsius), P.compare("<=", celsius, 8.0))));
console.log(Python3.Text.ToText(parsed));
const back = translate(parsed, Python3.Expressions.DIALECT, E.DIALECT);
console.log(show(back));
console.log(Terms.same(back, in_range));

2.0 <= shipment.celsius and shipment.celsius <= 8.0


and(le(2.0, get(shipment, 'celsius')), le(get(shipment, 'celsius'), 8.0))


true


Round trips keep what both languages can say. What one of them can't, such as a let in MATLAB, which has none, is
written another way, and comes back in that other form.

<a id="T14.4"></a>
## 14.4 · What a constraint may reach

A Python or MATLAB expression can call functions, and an expression loaded from a file mustn't be able to call
anything it likes. Each dialect's scope reaches only what the caller gives it, and an import it doesn't allow stays in the residual,
unresolved: here, a model of Python's `math`, and
nothing else:

In [7]:
const rounded = store.transaction(() =>
  Expression((e: any) => e.root(P.import_("math", P.compare("<=", P.call(P.attribute(P.name("math"), "floor"), celsius), 8n)))));
const kept = store.transaction(() => Expression((e: any) => e.root(Python3.Simplifiers.OfAny(rounded, { shipment: cold }).residual)));
console.log(Python3.Text.ToText(rounded));
console.log(Python3.Text.ToText(kept) === Python3.Text.ToText(rounded) ? "True" : "False"); // math not allowed: the import stays, unresolved
const math = new Python3.Simplifiers.Module("math", { floor: (x: number) => BigInt(Math.floor(x)) });
console.log(Python3.Simplifiers.OfAny(rounded, new Python3.Simplifiers.Scope({ shipment: cold }, { modules: { math } })).evaluate());

import math
math.floor(shipment.celsius) <= 8


ImportError: import of 'math' is not allowed by the scope


true


## Why it works this way

- **One translator per pair, by rules.** Each pair of dialects has its own translator, declared as rules pairing
  patterns, as the rewriting of case study 13 was. Translating directly or through Patterns gives the same result
  ([CAP-101: translation between dialects](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-101.md#translators)).
- **Form, not value.** A translator can't make Excel treat a missing value as unknown, and shouldn't try: the
  translated constraint must read the way an engineer in that language would write it.
- **Nothing is reachable by default.** Constraints travel as data, sometimes from people you don't fully trust, so a
  scope is an allowlist
  ([CAP-100: six languages' expression dialects](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-100.md#dialects)).

<!-- nav -->
[← 13 · Analyzing and rewriting constraints](13_Analyzing_And_Rewriting_Constraints.ipynb) · [Home](../../README.md) · [15 · Constraints for firmware and hardware →](15_Constraints_For_Firmware_And_Hardware.ipynb)